In [10]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [11]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("parsalatifi/s-and-p-500-us-stock-market-data")



DATASET_PATH = Path(path)
df = pd.read_csv(DATASET_PATH / "master_stock_data.csv")

In [12]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [13]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [14]:
df.head(10)

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,1999-11-18,A,26.189770,31.473534,0.0,35.765381,28.612303,32.546494,0.0,62546380
1,1999-11-19,A,24.032093,28.880545,0.0,30.758226,28.478184,30.713518,0.0,15234146
2,1999-11-22,A,26.189770,31.473534,0.0,31.473534,28.657009,29.551144,0.0,6577870
3,1999-11-23,A,23.808886,28.612303,0.0,31.205294,28.612303,30.400572,0.0,5975611
4,1999-11-24,A,24.441307,29.372318,0.0,29.998213,28.612303,28.701717,0.0,4843231
5,1999-11-26,A,24.515709,29.461731,0.0,29.685265,29.148785,29.238197,0.0,1729466
6,1999-11-29,A,25.073727,30.132332,0.0,30.355865,29.014664,29.327612,0.0,4074751
7,1999-11-30,A,25.110933,30.177038,0.0,30.713518,29.282904,30.042917,0.0,4310034
8,1999-12-01,A,25.557346,30.713518,0.0,31.071173,29.953505,30.177038,0.0,2957329
9,1999-12-02,A,26.264172,31.562946,0.0,32.188843,30.892345,31.294706,0.0,3069868


In [15]:
df.drop(columns=["Ticker","Dividends","Stock Splits","Adj Close","Date"],inplace=True)

In [16]:
print(df.isna().sum())

Close     4
High      4
Low       4
Open      4
Volume    0
dtype: int64


In [17]:
df = df.dropna()

In [18]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=["Close"]),df.iloc[:,:1],test_size=0.2)

In [19]:
x_scaler=StandardScaler()
x_train=x_scaler.fit_transform(x_train)
x_test=x_scaler.transform(x_test)

In [20]:
y_scaler=StandardScaler()
y_train=y_scaler.fit_transform(y_train)
y_test=y_scaler.transform(y_test)

In [21]:
#create CustomDataset Class
class CustomDataset(Dataset):
  def __init__(self, x, y):

    self.x = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.float32)

  def __len__(self):

    return len(self.x)

  def __getitem__(self, index):

    return self.x[index], self.y[index]


In [22]:
train_dataset = CustomDataset(x_train, y_train)

In [23]:
test_dataset = CustomDataset(x_test, y_test)

In [24]:
train_dataset[:10]

(tensor([[-0.4566, -0.4563, -0.4564, -0.0265],
         [ 3.0775,  3.0970,  3.0701,  0.1070],
         [-0.4696, -0.4695, -0.4696, -0.1252],
         [-0.2827, -0.2808, -0.2830,  0.2662],
         [ 0.3167,  0.3058,  0.3184, -0.1733],
         [-0.4282, -0.4277, -0.4277, -0.0352],
         [ 0.7217,  0.7128,  0.7238, -0.1583],
         [ 0.1970,  0.1957,  0.2037, -0.1454],
         [ 2.0599,  1.9557,  2.0880, -0.1728],
         [ 0.0154,  0.0173,  0.0202,  0.0595]]),
 tensor([[-0.4563],
         [ 3.1102],
         [-0.4695],
         [-0.2816],
         [ 0.2974],
         [-0.4282],
         [ 0.7019],
         [ 0.1915],
         [ 1.9585],
         [ 0.0131]]))

In [25]:
neurons_per_layer=24
epochs=15
learning_rate=1.2392889074215176e-05
dropout_rate = 0.2
batch_size= 128
optimizer= 'SGD'
weight_decay = 1.145060732085222e-05

In [26]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [27]:
class MyNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.model=nn.Sequential(
        nn.Linear(num_features,neurons_per_layer),
        nn.BatchNorm1d(neurons_per_layer),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(neurons_per_layer,neurons_per_layer),
        nn.BatchNorm1d(neurons_per_layer),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(neurons_per_layer,1)
    )

  def forward(self,x):

      return self.model(x)

In [28]:
x_train.shape[1]

4

In [29]:
model=MyNN(x_train.shape[1])
model=model.to(device)
criterion=nn.MSELoss()

optimizer=optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)

In [30]:
for epoch in range(epochs):
  model.train()
  total_epoch_loss=0
  for x_train_batch,y_train_batch in train_loader:
      x_train_batch,y_train_batch=x_train_batch.to(device),y_train_batch.to(device)
      outputs=model(x_train_batch)
      loss=criterion(outputs,y_train_batch)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      total_epoch_loss = total_epoch_loss + loss.item()
  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 0.35595428996990564
Epoch: 2 , Loss: 0.2897866294905033
Epoch: 3 , Loss: 0.2712201899591427
Epoch: 4 , Loss: 0.26227142982929247
Epoch: 5 , Loss: 0.2542835959217702
Epoch: 6 , Loss: 0.24842774771435344
Epoch: 7 , Loss: 0.24256817648650625
Epoch: 8 , Loss: 0.23792823802072635
Epoch: 9 , Loss: 0.23156028696678427
Epoch: 10 , Loss: 0.22964764943101082
Epoch: 11 , Loss: 0.2274685156730544
Epoch: 12 , Loss: 0.2253065586433401
Epoch: 13 , Loss: 0.22096097626526182
Epoch: 14 , Loss: 0.2187263088683208
Epoch: 15 , Loss: 0.21704661398564828


In [31]:
model.eval()
preds, targets = [], []

with torch.no_grad():
    for x_test_batch, y_test_batch in test_loader:
        x_test_batch = x_test_batch.to(device)
        outputs = model(x_test_batch)
        preds.append(outputs.cpu())
        targets.append(y_test_batch)

y_pred = torch.cat(preds).numpy().ravel()
y_true = torch.cat(targets).numpy().ravel()

mse  = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print(f'MSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}')

MSE: 0.0257 | RMSE: 0.1604 | MAE: 0.0950 | R2: 0.9734


In [32]:
y_pred

array([ 0.6495475 , -0.40960786,  0.5631698 , ..., -0.36467305,
       -0.3170186 , -0.32540274], shape=(443935,), dtype=float32)

In [33]:
y_true

array([ 0.9276346 , -0.4585641 ,  0.82181966, ..., -0.4227978 ,
       -0.336506  , -0.27907342], shape=(443935,), dtype=float32)